# ChromaDB Basics — a playground

This notebook has **nothing to do with the ML book** — it's just a small sandbox to
understand what ChromaDB is and how its basic operations work, before seeing it used
for real in `01_Build_Vector_Database.ipynb`.

**What is ChromaDB?** It's a *vector database*: a place to store text alongside its
*embedding* (a list of numbers representing meaning), so you can later search "which
stored text is closest in meaning to this new text?" instead of searching for exact
keyword matches.

We reuse the same OpenAI embedding setup as the other notebooks (it's already proven to
work in this project), instead of Chroma's built-in embedding model.

## 1. Setup

Same API key loading as the other notebooks, plus one small helper function to turn text
into an embedding. We'll reuse `embed_texts()` for every step below.

In [3]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(r"E:\Imarticus_ Learning\Jobs - 2026\Models\.env")
openai_client = OpenAI()
EMBEDDING_MODEL = "text-embedding-3-small"

def embed_texts(texts):
    """Turn a list of strings into a list of embedding vectors."""
    response = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [item.embedding for item in response.data]

## 2. Create a client and a collection

A **client** is your connection to Chroma. A **collection** is like a single table/folder
inside it, holding a set of related documents.

We use `chromadb.Client()` here (in-memory, resets every time we restart) so this
playground never touches the real database used by the other notebooks.

In [4]:
import chromadb

chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="playground")
collection

Collection(name=playground)

## 3. Add some documents

Every item needs a unique `id`. We embed the text ourselves with `embed_texts()`, then
hand both the text and its embedding to Chroma.

In [5]:
documents = [
    "The cat sat on the mat.",
    "Dogs are loyal and friendly animals.",
    "Paris is the capital of France.",
    "The Eiffel Tower is located in Paris.",
]

collection.add(
    ids=["id1", "id2", "id3", "id4"],
    documents=documents,
    embeddings=embed_texts(documents),
)

## 4. Count — how many items are stored?

In [6]:
collection.count()

4

## 5. Peek — look at a few raw stored records

`peek()` shows you the ids, the documents, and the embedding vector for each one (a long
list of numbers — we only see the first few here).

In [7]:
collection.peek(limit=2)

{'ids': ['id1', 'id2'],
 'embeddings': array([[-0.02442932, -0.03741455,  0.00107002, ...,  0.00434494,
         -0.029953  , -0.00306511],
        [ 0.01582336, -0.00252533, -0.00577164, ..., -0.00285149,
         -0.01899719,  0.0222168 ]], shape=(2, 1536)),
 'documents': ['The cat sat on the mat.',
  'Dogs are loyal and friendly animals.'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [None, None]}

## 6. Get — fetch specific items by id

Unlike `query()` (next section), `get()` does **not** rank by similarity — it just looks
items up directly, the same way a dictionary lookup would.

In [8]:
collection.get(ids=["id1", "id3"])

{'ids': ['id1', 'id3'],
 'embeddings': None,
 'documents': ['The cat sat on the mat.', 'Paris is the capital of France.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [None, None]}

## 7. Query — search by meaning

This is the core operation of a vector database. We embed some new text the same way we
embedded our documents, then ask Chroma for the closest matches ranked by **distance**
(smaller distance = more similar meaning).

Notice the question below shares almost no exact words with the matching document, yet
Chroma still finds it — because it searches by *meaning*, not keywords.

In [7]:
question = "Where is the Eiffel Tower?"

collection.query(query_embeddings=embed_texts([question]), n_results=2)

{'ids': [['id4', 'id3']],
 'embeddings': None,
 'documents': [['The Eiffel Tower is located in Paris.',
   'Paris is the capital of France.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None]],
 'distances': [[0.42475008964538574, 1.1876269578933716]]}

## 8. Metadata — attach extra facts to each item

Metadata is a small dictionary of extra info attached to each item (e.g. a category, a
page number, a date). We can then **filter** searches to only consider items whose
metadata matches, using `where=`.

In [8]:
more_documents = ["Sharks are fish, not mammals.", "Whales are mammals, not fish."]

collection.add(
    ids=["id5", "id6"],
    documents=more_documents,
    embeddings=embed_texts(more_documents),
    metadatas=[{"topic": "animals"}, {"topic": "animals"}],
)

In [9]:
collection.query(
    query_embeddings=embed_texts(["Tell me about the ocean"]),
    n_results=5,
    where={"topic": "animals"},
)

{'ids': [['id6', 'id5']],
 'embeddings': None,
 'documents': [['Whales are mammals, not fish.',
   'Sharks are fish, not mammals.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'topic': 'animals'}, {'topic': 'animals'}]],
 'distances': [[1.60616135597229, 1.648474097251892]]}

## 9. Update — change an existing item

Updating an id replaces its text and embedding, so future searches reflect the new
content.

In [10]:
new_text = "The cat is sleeping on the sofa."
collection.update(ids=["id1"], documents=[new_text], embeddings=embed_texts([new_text]))
collection.get(ids=["id1"])

{'ids': ['id1'],
 'embeddings': None,
 'documents': ['The cat is sleeping on the sofa.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [None]}

## 10. Delete — remove an item

In [11]:
collection.delete(ids=["id2"])
collection.count()

5

## 11. Persistence — saving to disk

`chromadb.Client()` (what we used above) is **in-memory only** — everything disappears
once the notebook's Python process ends. That's fine for this playground, but our real
notebooks need the database to survive between sessions.

For that, we use `PersistentClient(path=...)` instead — it writes the database to a
folder on disk. This is exactly what `01_Build_Vector_Database.ipynb` uses to save the
book's chunks permanently to `../chroma_db`.

In [12]:
persistent_client = chromadb.PersistentClient(path="../chroma_db_playground")
persistent_collection = persistent_client.get_or_create_collection(name="demo")

text = "This survives after the notebook closes."
persistent_collection.add(ids=["p1"], documents=[text], embeddings=embed_texts([text]))
persistent_collection.count()

1

Re-run the cell above after restarting the kernel: `count()` still returns 1, because the
data was written to the `chroma_db_playground/` folder on disk, not just kept in memory.

## Recap

| Operation | What it does |
|---|---|
| `create_collection` / `get_or_create_collection` | make or open a collection |
| `add` | store new items (text + embedding + optional metadata) |
| `count` | how many items are stored |
| `peek` | look at a few raw stored records |
| `get` | fetch specific items by id (exact lookup, no ranking) |
| `query` | search by meaning, ranked by distance |
| `where=` | filter `get`/`query` by metadata |
| `update` | replace an item's text/embedding/metadata |
| `delete` | remove an item |
| `Client()` vs `PersistentClient(path=...)` | in-memory (temporary) vs saved to disk |

That's the whole toolkit. Now open **`01_Build_Vector_Database.ipynb`** to see these same
operations (`PersistentClient`, `add`, `query`) used on the real ML book.